### Gold Layer - Star Schema

This notebook creates the business-ready Star Schema from the Silver layer for analytics and Power BI reporting.

In [0]:
storage_account_name = "stsupplychainlak"

spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
    dbutils.secrets.get(
        scope="adls-scope",
        key="storage-account-key"
    )
)

In [0]:
silver_df = spark.read.format("delta").load(
    "abfss://silver@stsupplychainlak.dfs.core.windows.net/yellow_taxi"
)

In [0]:
display(silver_df.limit(25))

DOLocationID,PULocationID,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee,pickup_year,pickup_month,pickup_day,pickup_hour,trip_duration_minutes,Pickup_Borough,Pickup_Zone,Pickup_Service_Zone,Dropoff_Borough,Dropoff_Zone,Dropoff_Service_Zone
74,138,1,2025-07-01T00:29:37,2025-07-01T00:45:30,1,7.3,1,N,1,29.6,7.75,0.5,9.0,6.94,1.0,54.79,0.0,1.75,0.0,2025,7,1,0,15.88,Queens,LaGuardia Airport,Airports,Manhattan,East Harlem North,Boro Zone
142,132,1,2025-07-01T00:23:28,2025-07-01T01:07:44,1,17.7,2,N,1,70.0,4.25,0.5,5.0,0.0,1.0,80.75,2.5,1.75,0.0,2025,7,1,0,44.27,Queens,JFK Airport,Airports,Manhattan,Lincoln Square East,Yellow Zone
48,138,2,2025-07-01T00:53:50,2025-07-01T01:27:12,1,9.98,1,N,1,43.6,6.0,0.5,10.87,0.0,1.0,66.97,2.5,1.75,0.75,2025,7,1,0,33.37,Queens,LaGuardia Airport,Airports,Manhattan,Clinton East,Yellow Zone
229,138,2,2025-07-01T00:58:49,2025-07-01T01:15:55,1,10.27,1,N,1,38.7,6.0,0.5,14.1,6.94,1.0,72.24,2.5,1.75,0.75,2025,7,1,0,17.1,Queens,LaGuardia Airport,Airports,Manhattan,Sutton Place/Turtle Bay North,Yellow Zone
97,211,2,2025-07-01T00:09:22,2025-07-01T00:23:54,1,2.94,1,N,1,17.0,1.0,0.5,3.0,0.0,1.0,25.75,2.5,0.0,0.75,2025,7,1,0,14.53,Manhattan,SoHo,Yellow Zone,Brooklyn,Fort Greene,Boro Zone
155,132,1,2025-07-01T00:39:14,2025-07-01T00:55:21,1,11.8,1,N,1,44.3,1.0,0.5,14.05,0.0,1.0,60.85,0.0,0.0,0.0,2025,7,1,0,16.12,Queens,JFK Airport,Airports,Brooklyn,Marine Park/Mill Basin,Boro Zone
263,79,2,2025-07-01T00:15:26,2025-07-01T00:29:39,1,3.87,1,N,1,17.7,1.0,0.5,4.69,0.0,1.0,28.14,2.5,0.0,0.75,2025,7,1,0,14.22,Manhattan,East Village,Yellow Zone,Manhattan,Yorkville West,Yellow Zone
262,140,2,2025-07-01T00:40:58,2025-07-01T00:44:15,1,0.85,1,N,1,5.8,1.0,0.5,2.16,0.0,1.0,12.96,2.5,0.0,0.0,2025,7,1,0,3.28,Manhattan,Lenox Hill East,Yellow Zone,Manhattan,Yorkville East,Yellow Zone
50,114,2,2025-07-01T00:28:12,2025-07-01T00:39:49,2,2.54,1,N,1,14.2,1.0,0.5,3.99,0.0,1.0,23.94,2.5,0.0,0.75,2025,7,1,0,11.62,Manhattan,Greenwich Village South,Yellow Zone,Manhattan,Clinton West,Yellow Zone
197,132,2,2025-07-01T00:38:17,2025-07-01T00:55:44,1,6.37,1,N,1,26.8,1.0,0.5,5.86,0.0,1.0,36.91,0.0,1.75,0.0,2025,7,1,0,17.45,Queens,JFK Airport,Airports,Queens,Richmond Hill,Boro Zone


### Create Date Dimension from the Silver dataset.

In [0]:
dim_date = silver_df.select(
    "pickup_year",
    "pickup_month",
    "pickup_day",
    "pickup_hour"
).distinct()

In [0]:
display(dim_date.limit(5))

pickup_year,pickup_month,pickup_day,pickup_hour
2025,7,1,0
2025,7,1,4
2025,7,1,1
2025,7,1,6
2025,7,1,3


### Create Pickup Zone Dimension.

In [0]:
dim_pickup_zone = silver_df.select(
    "PULocationID",
    "Pickup_Borough",
    "Pickup_Zone",
    "Pickup_Service_Zone"
).distinct()

In [0]:
display(dim_pickup_zone.limit(25))

PULocationID,Pickup_Borough,Pickup_Zone,Pickup_Service_Zone
119,Bronx,Highbridge,Boro Zone
264,Unknown,N/A,N/A
70,Queens,East Elmhurst,Boro Zone
80,Brooklyn,East Williamsburg,Boro Zone
186,Manhattan,Penn Station/Madison Sq West,Yellow Zone
116,Manhattan,Hamilton Heights,Boro Zone
158,Manhattan,Meatpacking/West Village West,Yellow Zone
100,Manhattan,Garment District,Yellow Zone
215,Queens,South Jamaica,Boro Zone
87,Manhattan,Financial District North,Yellow Zone


### Create Dropoff Zone Dimension.

In [0]:
dim_dropoff_zone = silver_df.select(
    "DOLocationID",
    "Dropoff_Borough",
    "Dropoff_Zone",
    "Dropoff_Service_Zone"
).distinct()

In [0]:
display(dim_dropoff_zone.limit(25))

DOLocationID,Dropoff_Borough,Dropoff_Zone,Dropoff_Service_Zone
217,Brooklyn,South Williamsburg,Boro Zone
119,Bronx,Highbridge,Boro Zone
247,Bronx,West Concourse,Boro Zone
154,Brooklyn,Marine Park/Floyd Bennett Field,Boro Zone
264,Unknown,N/A,N/A
255,Brooklyn,Williamsburg (North Side),Boro Zone
70,Queens,East Elmhurst,Boro Zone
80,Brooklyn,East Williamsburg,Boro Zone
46,Bronx,City Island,Boro Zone
186,Manhattan,Penn Station/Madison Sq West,Yellow Zone


### Create Payment Type Dimension.

In [0]:
from pyspark.sql import Row

payment_data = [
    Row(payment_type=0, payment_description="Flex Fare"),
    Row(payment_type=1, payment_description="Credit Card"),
    Row(payment_type=2, payment_description="Cash"),
    Row(payment_type=3, payment_description="No Charge"),
    Row(payment_type=4, payment_description="Dispute")
]

dim_payment_type = spark.createDataFrame(payment_data)

In [0]:
display(dim_payment_type.limit(25))

payment_type,payment_description
0,Flex Fare
1,Credit Card
2,Cash
3,No Charge
4,Dispute


### Create Fact Trips table for business analytics.

In [0]:
fact_trips = silver_df.select(
    
    # Date Keys
    "pickup_year",
    "pickup_month",
    "pickup_day",
    "pickup_hour",

    # Dimension Keys
    "PULocationID",
    "DOLocationID",
    "payment_type",

    # Additional Business Keys
    "VendorID",
    "RatecodeID",

    # Measures
    "passenger_count",
    "trip_distance",
    "fare_amount",
    "extra",
    "mta_tax",
    "tip_amount",
    "tolls_amount",
    "improvement_surcharge",
    "total_amount",
    "congestion_surcharge",
    "Airport_fee",
    "cbd_congestion_fee",
    "trip_duration_minutes"
)

In [0]:
display(fact_trips.limit(25))

pickup_year,pickup_month,pickup_day,pickup_hour,PULocationID,DOLocationID,payment_type,VendorID,RatecodeID,passenger_count,trip_distance,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee,trip_duration_minutes
2025,7,1,0,138,74,1,1,1,1,7.3,29.6,7.75,0.5,9.0,6.94,1.0,54.79,0.0,1.75,0.0,15.88
2025,7,1,0,132,142,1,1,2,1,17.7,70.0,4.25,0.5,5.0,0.0,1.0,80.75,2.5,1.75,0.0,44.27
2025,7,1,0,138,48,1,2,1,1,9.98,43.6,6.0,0.5,10.87,0.0,1.0,66.97,2.5,1.75,0.75,33.37
2025,7,1,0,138,229,1,2,1,1,10.27,38.7,6.0,0.5,14.1,6.94,1.0,72.24,2.5,1.75,0.75,17.1
2025,7,1,0,211,97,1,2,1,1,2.94,17.0,1.0,0.5,3.0,0.0,1.0,25.75,2.5,0.0,0.75,14.53
2025,7,1,0,132,155,1,1,1,1,11.8,44.3,1.0,0.5,14.05,0.0,1.0,60.85,0.0,0.0,0.0,16.12
2025,7,1,0,79,263,1,2,1,1,3.87,17.7,1.0,0.5,4.69,0.0,1.0,28.14,2.5,0.0,0.75,14.22
2025,7,1,0,140,262,1,2,1,1,0.85,5.8,1.0,0.5,2.16,0.0,1.0,12.96,2.5,0.0,0.0,3.28
2025,7,1,0,114,50,1,2,1,2,2.54,14.2,1.0,0.5,3.99,0.0,1.0,23.94,2.5,0.0,0.75,11.62
2025,7,1,0,132,197,1,2,1,1,6.37,26.8,1.0,0.5,5.86,0.0,1.0,36.91,0.0,1.75,0.0,17.45


### Store the Star Schema tables in the Gold layer.

In [0]:
# Write DIM_DATE
dim_date.write \
.format("delta") \
.mode("overwrite") \
.save(
    "abfss://gold@stsupplychainlak.dfs.core.windows.net/dim_date"
)

In [0]:
# Write DIM_PICKUP_ZONE
dim_pickup_zone.write \
.format("delta") \
.mode("overwrite") \
.save(
    "abfss://gold@stsupplychainlak.dfs.core.windows.net/dim_pickup_zone"
)

In [0]:
# Write DIM_DROPOFF_ZONE
dim_dropoff_zone.write \
.format("delta") \
.mode("overwrite") \
.save(
    "abfss://gold@stsupplychainlak.dfs.core.windows.net/dim_dropoff_zone"
)

In [0]:
# Write DIM_PAYMENT_TYPE
dim_payment_type.write \
.format("delta") \
.mode("overwrite") \
.save(
    "abfss://gold@stsupplychainlak.dfs.core.windows.net/dim_payment_type"
)

In [0]:
# Write FACT_TRIPS
fact_trips.write \
.format("delta") \
.mode("overwrite") \
.save(
    "abfss://gold@stsupplychainlak.dfs.core.windows.net/fact_trips"
)

### Validate

In [0]:
gold_fact = spark.read.format("delta").load(
    "abfss://gold@stsupplychainlak.dfs.core.windows.net/fact_trips"
)

display(gold_fact.limit(10))

pickup_year,pickup_month,pickup_day,pickup_hour,PULocationID,DOLocationID,payment_type,VendorID,RatecodeID,passenger_count,trip_distance,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee,trip_duration_minutes
2025,7,1,0,138,74,1,1,1,1,7.3,29.6,7.75,0.5,9.0,6.94,1.0,54.79,0.0,1.75,0.0,15.88
2025,7,1,0,132,142,1,1,2,1,17.7,70.0,4.25,0.5,5.0,0.0,1.0,80.75,2.5,1.75,0.0,44.27
2025,7,1,0,138,48,1,2,1,1,9.98,43.6,6.0,0.5,10.87,0.0,1.0,66.97,2.5,1.75,0.75,33.37
2025,7,1,0,138,229,1,2,1,1,10.27,38.7,6.0,0.5,14.1,6.94,1.0,72.24,2.5,1.75,0.75,17.1
2025,7,1,0,211,97,1,2,1,1,2.94,17.0,1.0,0.5,3.0,0.0,1.0,25.75,2.5,0.0,0.75,14.53
2025,7,1,0,132,155,1,1,1,1,11.8,44.3,1.0,0.5,14.05,0.0,1.0,60.85,0.0,0.0,0.0,16.12
2025,7,1,0,79,263,1,2,1,1,3.87,17.7,1.0,0.5,4.69,0.0,1.0,28.14,2.5,0.0,0.75,14.22
2025,7,1,0,140,262,1,2,1,1,0.85,5.8,1.0,0.5,2.16,0.0,1.0,12.96,2.5,0.0,0.0,3.28
2025,7,1,0,114,50,1,2,1,2,2.54,14.2,1.0,0.5,3.99,0.0,1.0,23.94,2.5,0.0,0.75,11.62
2025,7,1,0,132,197,1,2,1,1,6.37,26.8,1.0,0.5,5.86,0.0,1.0,36.91,0.0,1.75,0.0,17.45


In [0]:
spark.sql("""
CREATE SCHEMA IF NOT EXISTS
databricks_supplychain_dev.gold
""")

DataFrame[]

In [0]:
fact_trips.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable(
    "databricks_supplychain_dev.gold.fact_trips"
)

In [0]:
dim_date.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable(
    "databricks_supplychain_dev.gold.dim_date"
)

In [0]:
dim_pickup_zone.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable(
    "databricks_supplychain_dev.gold.dim_pickup_zone"
)

In [0]:
dim_dropoff_zone.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable(
    "databricks_supplychain_dev.gold.dim_dropoff_zone"
)

In [0]:
dim_payment_type.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable(
    "databricks_supplychain_dev.gold.dim_payment_type"
)